<a href="https://colab.research.google.com/github/Sebi2005/Metaheuristics/blob/main/notebooks/Steepest_Ascent_Hill_Climbing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1. Problem defintion**

The 0/1 Knapsack Problem is a combinatorial optimization problem. Given a set of items, each item has an associated weight and value. The objective is to select a subset of items such that the total value is maximized while the total weight does not exceed the capacity of the knapsack.

Formally:

Each item i has weight wi and value vi.
	​

The knapsack has capacity C.

A solution is represented as a binary vector: x=(x1,x2,...,xn) where:

xi = 1 if the item in included

xi = 0 if the item is not included

The objective is to maximize the sum of values subject to the capacity.

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving knapsack-20.txt to knapsack-20.txt
Saving knapsack-200.txt to knapsack-200.txt


In [ ]:
import random
def load_data_with_capacity(file_name:str):
  with open(file_name) as f:
    lines = [line.strip() for line in f if line.strip()]
  n = int(lines[0])
  items = []
  for i in range(1,n+1):
    parts = lines[i].split()
    weight = int(parts[1])
    value = int(parts[2])
    items.append((weight,value))

  capacity = int(lines[n+1])
  return items,capacity

load_data_with_capacity("knapsack-20.txt")

([(91, 29),
  (60, 65),
  (61, 71),
  (9, 60),
  (79, 45),
  (46, 71),
  (19, 22),
  (57, 97),
  (8, 6),
  (84, 91),
  (20, 57),
  (72, 60),
  (32, 49),
  (31, 89),
  (28, 2),
  (81, 30),
  (55, 90),
  (43, 25),
  (100, 82),
  (27, 19)],
 524)

**2. How solutions are evaluated**

The fitness of a solution is defined as the total value of selected items(the ones with value 1).


If the total weight exceeds the capacity of the knapsack, the solution is considered infeasible and its fitness is set to 0.

In [ ]:
def evaluate(solution, items, capacity):
  total_weight = 0
  total_value = 0
  for bit, (weight, value) in zip(solution, items):
    if bit == 1:
      total_weight += weight
      total_value += value
  if total_weight > capacity :
    return 0
  return total_value

In [ ]:
def random_solution(n):
  return [random.randint(0,1) for _ in range(n)]

**3. Algorithm used**

**Steepest Ascent Hill Climbing**

Steepest Ascent Hill Climbing is a local search algorithm that iteratively improves a solution by exploring its neighborhood.

A solution is represented as a binary string of length n, where each bit indicates whether an item is included in the knapsack.

The algorithm works as follows:

Generate a random binary solution.

Evaluate its fitness.

Generate all neighboring solutions by flipping one bit at a time.

Evaluate all neighbors.

Select the neighbor with the highest fitness.

If the neighbor improves the current solution, move to that neighbor.

If no improvement is possible, a local optimum has been reached and the algorithm restarts with a new random solution.

The algorithm stops when the maximum number of function evaluations is reached.

The best solution found during the search is returned.

In [ ]:
def steepest_ascent_hill_climbing(items, capacity, max_evals):
  n = len(items)
  current = random_solution(n)
  current_fitness = evaluate(current, items, capacity)

  best = current[:]
  best_fitness = current_fitness

  evaluations = 1
  while evaluations < max_evals:
    best_neighbor = None
    best_neighbor_fitness = -1
    for i in range(n):
      neighbor = current[:]
      neighbor[i] = 1- neighbor[i]
      neighbor_fitness = evaluate(neighbor, items, capacity)

      evaluations +=1
      if neighbor_fitness > best_neighbor_fitness:
        best_neighbor_fitness = neighbor_fitness
        best_neighbor = neighbor

      if evaluations >= max_evals:
        break
    if best_neighbor_fitness > current_fitness:
      current_fitness = best_neighbor_fitness
      current = best_neighbor

      if current_fitness > best_fitness:
        best_fitness = current_fitness
        best = current[:]
    else:
      current = random_solution(n)
      current_fitness = evaluate(current,items,capacity)
      evaluations += 1
  return best, best_fitness






In [ ]:
import time

def run_experiment(file_name, eval_values, runs_per_k=10, output_file=None):

    items, capacity = load_data_with_capacity(file_name)

    results = []

    if output_file:
        f = open(output_file, "w")

    for k in eval_values:
        if output_file:
            f.write(f"\nParameter max_evals = {k}\n")
        for run in range(1, runs_per_k + 1):
            start = time.time()
            solution, best_value = steepest_ascent_hill_climbing(
                items,
                capacity,
                max_evals=k
            )
            end = time.time()
            time_used = end - start
            results.append((k, run, best_value, time_used))
            if output_file:
                f.write(
                    f"run {run}  value={best_value}  time={time_used:.5f}\n"
                )

    if output_file:
        f.close()

    return results

In [ ]:
def summarize_results(results, output_file=None):

    groups = {}

    for k, run, value, time_used in results:

        if k not in groups:
            groups[k] = {
                "values": [],
                "times": []
            }

        groups[k]["values"].append(value)
        groups[k]["times"].append(time_used)

    if output_file:
        f = open(output_file, "w")

    for k in sorted(groups):

        values = groups[k]["values"]
        times = groups[k]["times"]

        avg_value = sum(values) / len(values)
        best_value = max(values)
        worst_value = min(values)
        avg_time = sum(times) / len(times)

        line = (
            f"max_evals={k} | "
            f"avg_value={avg_value:.2f} | "
            f"best={best_value} | "
            f"avg_time={avg_time:.5f}\n"
        )

        print(line)

        if output_file:
            f.write(line)

    if output_file:
        f.close()

**4. Parameter settings**

The algorithm was tested using different values for the maximum number of function evaluations. This parameter controls how long the algorithm runs.

Each configuration was executed 10 times in order to observe variability caused by the random initialization.

The tested parameter values were:

For knapsack-20:

100

200

500

1000

For knapsack-200:

1000

5000

10000

20000

The results of all sets were saved in output files

In [ ]:
results_20 = run_experiment(
    "knapsack-20.txt",
    [100,200,500,1000],
    runs_per_k=10,
    output_file="results_20.txt"
)
summarize_results(results_20, "summary_20.txt")
results_200 = run_experiment(
    "knapsack-200.txt",
    [1000,5000,10000,20000],
    runs_per_k=10,
    output_file="results_200.txt"
)
summarize_results(results_200, "summary_200.txt")

max_evals=100 | avg_value=636.70 | best=717 | worst=553 | avg_time=0.00050

max_evals=200 | avg_value=680.50 | best=756 | worst=594 | avg_time=0.00052

max_evals=500 | avg_value=727.10 | best=763 | worst=665 | avg_time=0.00104

max_evals=1000 | avg_value=746.20 | best=785 | worst=692 | avg_time=0.00215

max_evals=1000 | avg_value=28894.30 | best=96464 | worst=0 | avg_time=0.01694

max_evals=5000 | avg_value=57766.80 | best=96730 | worst=0 | avg_time=0.09398

max_evals=10000 | avg_value=86835.20 | best=97333 | worst=0 | avg_time=0.17055

max_evals=20000 | avg_value=96981.00 | best=97328 | worst=96572 | avg_time=0.33957



**5. Comparative Results of Experiments**

knapsack-20

|Evaluations | Avg value | Max value | Time |
|------------|-----------|-----------|------|
|100         | 595.40    | 717       | 0.0040|
|200         | 663.40    | 724       | 0.00062|
|500         |745.90     | 787       | 0.00112|
|1000        |745.90     | 779       | 0.00210|

knapsack-200
|Evaluations | Avg value | Max value | Time |
|------------|-----------|-----------|------|
|1000         | 57921.50    | 97101       | 0.01693|
|5000         | 86871.40    | 97117       | 0.08800|
|10000         |96459.00     | 97069       | 0.17232|
|20000        |96546.60     | 97077       | 0.47546|


**6. Discussion of Results**

The experimental results show that increasing the number of function evaluations generally improves the quality of the solution. With more evaluations, the algorithm explores more candidate solutions and has a higher probability of escaping poor local optima.

However, the improvement becomes smaller for large evaluation values. This occurs because the algorithm often reaches good local optima early in the search.

The results for the 200-item instance show greater variability compared to the 20-item instance, because the search space is significantly larger.

Since hill climbing is a greedy algorithm, it can become trapped in local optima. Random restarts help mitigate this issue by exploring different regions of the search space.

Overall, the Steepest Ascent Hill Climbing algorithm provides reasonably good solutions for the knapsack problem within a limited number of evaluations.